# 00a — GEO conversion: trunk morph counts and guide calls → `sample_284.adata`

**Feeds:** `01_preprocessing.ipynb`, and `00b_geo_human_embryo.ipynb`, which reads the object's gene IDs

**Position in the chain:** first. Run `00a`, then `00b`, then `01_preprocessing`.

Ported from the authors' notebook `day6_PDMS+glass+limbbud/d6_PDMS+glass+limbbud_SZ1.06.ipynb`, cells
0–5 (the notebook's cell list, counted from 0). Cell 0 imports; cells 1–4 read the guide calls and the
10x count matrix and attach each cell's guide call and guide count; cell 5 writes the object. The
notebook's later cells are a different analysis and are not ported.

**Input** — GEO series GSE306808, sample GSM9209686, both files in `data/GSE306808/` (or in
`$GEO_DATA_DIR/GSE306808/`):
- `GSM9209686_filtered_feature_bc_matrix.h5`
- `GSM9209686_barcode_data.csv.gz`

**Output** — `sample_284.adata`, written to `$SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`),
where `01_preprocessing` reads it.

**Changes from the original notebook**
1. The inputs are the GEO files. The original read `filtered_feature_bc_matrix.h5` and
   `barcode_284_data.csv` under their local names; GEO serves the same CSV gzip-compressed, and pandas
   reads the `.gz` directly.
2. Paths: the first code cell was added to locate the GEO files and the output directory. The original
   read and wrote in its working directory.

No other line of code was changed.

In [ ]:
from pathlib import Path
import os
import sys

# Paths. Added for this repository: the original notebook read and wrote in its working directory.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "trunk_morph_ref").is_dir())
# GEO downloads: GEO_DATA_DIR/GSE306808/ (default data/GSE306808/).
GEO_DATA_DIR = Path(os.environ.get("GEO_DATA_DIR", REPO / "data"))
# 01_preprocessing reads what this notebook writes from $SCRNASEQ_INPUT_ROOT (default data/scrnaseq_inputs/).
sys.path.insert(0, str(REPO))
from src.trunk_morph_ref.paths import scrnaseq_input_root
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO)
SCRNASEQ_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("reading GEO files from", GEO_DATA_DIR / "GSE306808")
print("writing to", SCRNASEQ_INPUT_ROOT)


In [ ]:
import scanpy as sc
import numpy as np
import scipy
import pandas as pd
import anndata as ad
from scipy.sparse import load_npz, save_npz, coo_matrix, csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read from CSV
barcode_284_data_csv = pd.read_csv(GEO_DATA_DIR / 'GSE306808' / 'GSM9209686_barcode_data.csv.gz')
print("Data from CSV:")
print(barcode_284_data_csv.head())
barcode_284_data_csv.shape

In [ ]:
# Store minimum dataframe
barcode_284_data_min = barcode_284_data_csv[['cell_barcode', 'guide', 'final_count']]

In [ ]:
# Load the cells_by_genes data into scanpy as Anndata

sample_284 = sc.read_10x_h5(GEO_DATA_DIR / 'GSE306808' / 'GSM9209686_filtered_feature_bc_matrix.h5')
sample_284.var_names_make_unique()

In [ ]:
# Extract cell barcodes from the cells_by_genes Anndata
cell_ids_284 = pd.DataFrame()
cell_ids_284['cell_barcode'] = sample_284.obs.index

# Merge the guide calls and counts to the cell barcodes, keeping the order from Anndata
merger_df_284 = cell_ids_284.merge(barcode_284_data_min, on='cell_barcode', how='left')

# Re-index merger dataframes by cell barcode
merger_df_284_reindexed = merger_df_284.set_index('cell_barcode')

# Assign new observations to the Anndata
sample_284.obs['guide'] = merger_df_284_reindexed['guide']
sample_284.obs['guide_count'] = merger_df_284_reindexed['final_count']

In [ ]:
sample_284.write_h5ad(SCRNASEQ_INPUT_ROOT / "sample_284.adata")